# Emergency Action Recognition: Google Colab GPU Workflow & Benchmark

This notebook provides the unified, reproducible Google Colab GPU workflow for **Emergency Vision AI**.

### Architecture & Artifact Strategy
- **Canonical Checkpoint**: Stored and tracked in the repository at `models/action_recognition/r3d18_urfd_best.pth` via Git LFS.
- **Canonical Detection Weights**: `models/detection/yolo11n.pt`.
- **Google Drive**: Used for persistent dataset storage (`/content/drive/MyDrive/emergency-vision-ai/data/urfd`) and backup/export.
- **Production Pipeline**: YOLO11n → ByteTrack → Per-Person 16-frame Rolling Buffer → R3D-18 Binary Action Classifier → Temporal Confirmation → `EmergencyActionEvent`.

## 1. Environment & Google Drive Setup

In [ ]:
# ==============================================================================
# CELL 1 — Environment & Google Drive Setup
# ==============================================================================
import os
import torch
from google.colab import drive

# 1. Mount Google Drive for dataset access & persistent backup
print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/emergency-vision-ai"
DRIVE_DATASET_DIR = os.path.join(DRIVE_ROOT, "data", "urfd")
DRIVE_BACKUP_CHECKPOINT = os.path.join(DRIVE_ROOT, "models", "action_recognition", "r3d18_urfd_best.pth")

print("=" * 60)
print("HARDWARE & ENVIRONMENT VERIFICATION")
print("=" * 60)
print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available in this Colab session!\n"
        "Please switch runtime: Runtime -> Change runtime type -> T4 GPU"
    )

print(f"GPU Accelerator:      {torch.cuda.get_device_name(0)}")
print(f"Drive Backup Checkpoint: {DRIVE_BACKUP_CHECKPOINT} (Exists: {os.path.exists(DRIVE_BACKUP_CHECKPOINT)})")
print(f"Drive Dataset Path:      {DRIVE_DATASET_DIR} (Exists: {os.path.exists(DRIVE_DATASET_DIR)})")
print("=" * 60)

## 2. Repository Clone / Sync & Source Verification

In [ ]:
# ==============================================================================
# CELL 2 — Repository Setup & Source File Integrity Verification
# ==============================================================================
import os
import sys

REPO_URL = "https://github.com/mukhammadiev01-1/emergency-vision-ai.git"
LOCAL_REPO_DIR = "/content/emergency-vision-ai"

if not os.path.exists(LOCAL_REPO_DIR):
    print(f"Cloning repository into {LOCAL_REPO_DIR}...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists at {LOCAL_REPO_DIR}. Pulling latest changes...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}

# Verify expected production files
REQUIRED_FILES = [
    "apps/worker/app/models/action_model.py",
    "apps/worker/app/models/yolo.py",
    "apps/worker/app/pipeline/tracking.py",
    "apps/worker/app/pipeline/action_recognition.py",
    "apps/worker/app/pipeline/events.py",
    "scripts/benchmark_gpu.py",
    "scripts/download_models.py",
    "requirements-worker.txt",
]

missing = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(LOCAL_REPO_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing required production files: {missing}")

print(f"Working Directory: {os.getcwd()}")
print("Repository source files verified successfully.")

## 3. Install Dependencies

In [ ]:
# ==============================================================================
# CELL 3 — Dependencies Installation & Version Verification
# ==============================================================================
import sys
import torch
import torchvision

print("Installing production dependencies...")
!pip install -q --upgrade pip
!pip install -q -r requirements-worker.txt
!pip install -q ultralytics certifi av

import ultralytics

print("=" * 60)
print("ENVIRONMENT DEPENDENCIES")
print("=" * 60)
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Torchvision:  {torchvision.__version__}")
print(f"Ultralytics:  {ultralytics.__version__}")
print("=" * 60)

## 4. Production Imports Verification

In [ ]:
# ==============================================================================
# CELL 4 — Production Imports Verification
# ==============================================================================
import sys
import os

LOCAL_REPO_DIR = "/content/emergency-vision-ai"
if LOCAL_REPO_DIR not in sys.path:
    sys.path.insert(0, LOCAL_REPO_DIR)
if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

print("Verifying actual production imports from repository...")

try:
    from apps.worker.app.models.action_model import (
        ActionRecognitionWrapper,
        ActionPrediction,
        preprocess_clip_frames,
    )
    from apps.worker.app.models.yolo import YOLOModelWrapper
    from apps.worker.app.pipeline.tracking import TrackingStage
    from apps.worker.app.pipeline.action_recognition import (
        ActionRecognitionStage,
        extract_person_crop,
        TrackActionState,
    )
    from apps.worker.app.pipeline.events import EmergencyActionEvent
    print("SUCCESS: Production modules imported cleanly.")
except ImportError as err:
    raise RuntimeError(f"FATAL: Production module import failed: {err}")

## 5. Canonical Checkpoint & Dataset Verification

In [ ]:
# ==============================================================================
# CELL 5 — Canonical Checkpoint & URFD Dataset Verification
# ==============================================================================
import hashlib
import os
import shutil

CANONICAL_CHECKPOINT = "models/action_recognition/r3d18_urfd_best.pth"
LOCAL_DATA_DIR = "data/urfd"

os.makedirs(os.path.dirname(CANONICAL_CHECKPOINT), exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)

# 1. Ensure canonical checkpoint is present in repository
if not os.path.exists(CANONICAL_CHECKPOINT) or os.path.getsize(CANONICAL_CHECKPOINT) < 1024 * 1024:
    if os.path.exists(DRIVE_BACKUP_CHECKPOINT) and os.path.getsize(DRIVE_BACKUP_CHECKPOINT) > 1024 * 1024:
        print(f"Restoring checkpoint from Google Drive backup: {DRIVE_BACKUP_CHECKPOINT} -> {CANONICAL_CHECKPOINT}...")
        shutil.copy2(DRIVE_BACKUP_CHECKPOINT, CANONICAL_CHECKPOINT)
    else:
        print("Pulling weights via Git LFS...")
        !git lfs pull --include="models/action_recognition/r3d18_urfd_best.pth"

# Compute SHA-256 hash
def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

ckpt_size_mb = os.path.getsize(CANONICAL_CHECKPOINT) / (1024 * 1024) if os.path.exists(CANONICAL_CHECKPOINT) else 0
ckpt_hash = compute_sha256(CANONICAL_CHECKPOINT) if os.path.exists(CANONICAL_CHECKPOINT) else 'N/A'

# 2. Link or download URFD dataset from Google Drive
if not os.path.exists(DRIVE_DATASET_DIR):
    print(f"Downloading URFD dataset to Drive: {DRIVE_DATASET_DIR}...")
    !python scripts/download_urfd.py --output-dir "{DRIVE_DATASET_DIR}" --format mp4

if os.path.islink(LOCAL_DATA_DIR):
    os.unlink(LOCAL_DATA_DIR)
elif os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)
os.symlink(DRIVE_DATASET_DIR, LOCAL_DATA_DIR)

fall_videos = sorted([f for f in os.listdir(os.path.join(LOCAL_DATA_DIR, "videos", "fall")) if f.endswith('.mp4')])
normal_videos = sorted([f for f in os.listdir(os.path.join(LOCAL_DATA_DIR, "videos", "normal")) if f.endswith('.mp4')])

print("=" * 70)
print("CANONICAL MODEL & DATASET VERIFICATION")
print("=" * 70)
print(f"Canonical Checkpoint Path: {CANONICAL_CHECKPOINT}")
print(f"Checkpoint File Size:      {ckpt_size_mb:.2f} MB")
print(f"Checkpoint SHA-256:        {ckpt_hash}")
print(f"FALL Videos (Class 1):     {len(fall_videos)} / 30")
print(f"NORMAL Videos (Class 0):   {len(normal_videos)} / 40")
print(f"Total URFD Sequences:      {len(fall_videos) + len(normal_videos)} / 70")
print("=" * 70)

if ckpt_size_mb < 50.0:
    raise ValueError(f"Invalid checkpoint size ({ckpt_size_mb:.2f} MB). Ensure the full .pth file is present.")

## 6. Action Model Smoke Test on CUDA

In [ ]:
# ==============================================================================
# CELL 6 — Action Model Smoke Test on CUDA
# ==============================================================================
import torch
from apps.worker.app.models.action_model import ActionRecognitionWrapper

print(f"Loading canonical R3D-18 checkpoint: {CANONICAL_CHECKPOINT} on CUDA...")
action_model = ActionRecognitionWrapper(
    weights_path=CANONICAL_CHECKPOINT,
    device="cuda",
    num_classes=2,
)
action_model._ensure_loaded()

# Run single 16-frame clip smoke test
dummy_clip_tensor = torch.randn(1, 3, 16, 112, 112, device="cuda", dtype=torch.float32)
torch.cuda.synchronize()
smoke_pred = action_model.predict_tensor(dummy_clip_tensor)
torch.cuda.synchronize()

print("=" * 60)
print("R3D-18 CUDA SMOKE TEST RESULT")
print("=" * 60)
print(f"Device:                {action_model.device}")
print(f"Input Tensor Shape:    {dummy_clip_tensor.shape} [B, C, T, H, W]")
print(f"Predicted Action:      {smoke_pred.action}")
print(f"Confidence:            {smoke_pred.confidence:.4f}")
print(f"FALL Probability:      {smoke_pred.fall_probability:.4f}")
print(f"NORMAL Probability:    {smoke_pred.normal_probability:.4f}")
print("=" * 60)

## 7. Model Evaluation on Held-Out Test Split

In [ ]:
# ==============================================================================
# CELL 7 — Model Evaluation on Held-Out Test Split (No Retraining)
# ==============================================================================
# Held-out test split (11 sequences: 5 Falls + 6 Normals, Seed=42)

print("=" * 60)
print("EVALUATING CANONICAL CHECKPOINT ON TEST SPLIT (Seed=42)")
print("=" * 60)

!python scripts/evaluate_action_model.py \
    --checkpoint "{CANONICAL_CHECKPOINT}" \
    --dataset-root "{LOCAL_DATA_DIR}" \
    --batch-size 4 \
    --device cuda \
    --seed 42

## 8. Canonical Production Pipeline GPU Benchmark

In [ ]:
# ==============================================================================
# CELL 8 — Canonical Production Pipeline GPU Benchmark
# ==============================================================================
# Runs the production benchmark script with CUDA synchronization:
# YOLO11n -> ByteTrack -> Per-Person 16-frame buffer -> R3D-18 -> Confirmation -> FALL Event

BENCHMARK_VIDEO = "data/urfd/videos/fall/fall-01-cam0.mp4"

!python scripts/benchmark_gpu.py \
    --video "{BENCHMARK_VIDEO}" \
    --action-model "{CANONICAL_CHECKPOINT}" \
    --yolo-model "models/detection/yolo11n.pt" \
    --device cuda \
    --threshold 0.70 \
    --interval 8 \
    --warmup 5

## 9. [Optional] Full Model Retraining Pipeline

The cell below is preserved for training history and reproducibility. You do NOT need to run this cell unless you want to train a new checkpoint from scratch.

In [ ]:
# [OPTIONAL] Two-Stage Transfer Learning from Scratch
print("=" * 60)
print("LAUNCHING OPTIONAL TWO-STAGE URFD R3D-18 RETRAINING")
print("=" * 60)

!python scripts/train_action_model.py \
    --dataset-root "{LOCAL_DATA_DIR}" \
    --output-dir "models/action_recognition" \
    --checkpoint-name "r3d18_urfd_best.pth" \
    --stage1-epochs 5 \
    --stage2-epochs 20 \
    --batch-size 4 \
    --device cuda \
    --seed 42

# Backup newly trained checkpoint to Google Drive
shutil.copy2("models/action_recognition/r3d18_urfd_best.pth", DRIVE_BACKUP_CHECKPOINT)
print(f"Backed up new checkpoint to Google Drive: {DRIVE_BACKUP_CHECKPOINT}")